# Домашнее задание 2 (часть 2): реализуем трансформер

В этой части задания вы поэтапно реализуете языковую модель на основе трансформера, а затем попробуете с её помощью сгенерировать связный текст.

In [2]:
%pip install -q transformers huggingface_hub


In [3]:
import math
import torch
import transformers

import numpy as np
import torch.nn as nn
import torch.nn.functional as F


In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [5]:
from huggingface_hub import hf_hub_download
state_dict = torch.load(hf_hub_download("gpt2", filename="pytorch_model.bin"))
for key, value in tuple(state_dict.items()):
    if key.startswith('h.') and key.endswith('.weight') and value.ndim == 2:
        value.transpose_(1, 0)  # для совместимости с современными модулями PyTorch
    if key.startswith('h.') and key.endswith('.attn.bias') and value.ndim == 4:
        state_dict.pop(key)  # треугольные бинарные маски в этой реализации не нужны

print('Веса:', repr(sorted(state_dict.keys()))[:320], '...')

Веса: ['h.0.attn.c_attn.bias', 'h.0.attn.c_attn.weight', 'h.0.attn.c_proj.bias', 'h.0.attn.c_proj.weight', 'h.0.ln_1.bias', 'h.0.ln_1.weight', 'h.0.ln_2.bias', 'h.0.ln_2.weight', 'h.0.mlp.c_fc.bias', 'h.0.mlp.c_fc.weight', 'h.0.mlp.c_proj.bias', 'h.0.mlp.c_proj.weight', 'h.1.attn.c_attn.bias', 'h.1.attn.c_attn.weight', 'h.1. ...


В следующих ячейках мы поэтапно реализуем модель, чтобы использовать загруженные веса.

Как вы помните, в трансформерах используются два основных типа блоков: блоки внимания и полносвязные блоки.

Полносвязный блок устроен проще, поэтому начнём с него.

Реализуйте полносвязный блок __без остаточных связей (residual connections) и нормализации слоя (LayerNorm)__ — мы добавим их чуть позже.

## Реализуйте полносвязный блок модели (2 балла)

In [6]:
class GeLUThatWasUsedInGPT2(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1.0 + torch.tanh(math.sqrt(2.0 / math.pi) * (x + 0.044715 * x ** 3)))

class FullyConnected(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.c_fc = nn.Linear(dim, 4  * dim)
        self.gelu = GeLUThatWasUsedInGPT2()
        self.c_proj = nn.Linear(4 * dim, dim)

    def forward(self, x):
        # x.shape = [batch_size, seq_length, dim]: размер батча, длина последовательности, размерность эмбеддинга
        hidden = self.c_fc(x)
        hidden = self.gelu(hidden)
        output = self.c_proj(hidden)
        return output

Проверим, что ваш блок корректно работает с весами GPT-2:

In [7]:
mlp = FullyConnected(dim=768)
mlp.load_state_dict({'c_fc.weight': state_dict['h.0.mlp.c_fc.weight'],
                     'c_fc.bias': state_dict['h.0.mlp.c_fc.bias'],
                     'c_proj.weight': state_dict['h.0.mlp.c_proj.weight'],
                     'c_proj.bias': state_dict['h.0.mlp.c_proj.bias']})

torch.manual_seed(1337)
x = torch.randn(1, 2, 768)  # [batch_size, sequence_length, dim]
checksum = torch.sum(mlp(x) * x)
assert abs(checksum.item() - 1282.3315) < 0.1, "Выход блока не совпадает с эталоном"
assert torch.allclose(mlp(x[:, (1, 0), :])[:, (1, 0), :], mlp(x)), "Перестановка входных позиций должна приводить к такой же перестановке выходов MLP"
print("Проверки пройдены!")

Проверки пройдены!


Теперь перейдём к блокам внимания.

GPT-2 генерирует текст слева направо, поэтому при вычислении представления каждого токена механизм внимания может учитывать только токены слева и сам этот токен. Такой механизм называется самовниманием с маской (masked self-attention): каузальная маска скрывает токены справа, то есть будущие токены.

Ниже приведён пример неэффективной реализации. Опираясь на материалы лекции и этот код, разработайте эффективную реализацию механизма самовнимания с маской.

In [8]:
class MaskedSelfAttentionSlow(nn.Module):
    def __init__(self, dim: int, num_heads: int):
        super().__init__()
        self.c_attn = nn.Linear(dim, dim * 3)  # совместное вычисление запросов (query), ключей (key) и значений (value)
        self.c_proj = nn.Linear(dim, dim)  # выходная проекция
        self.dim, self.num_heads = dim, num_heads
        self.head_size = dim // num_heads

    def forward(self, x):
        q, k, v = self.c_attn(x).split(dim=-1, split_size=self.dim)
        assert q.shape == k.shape == v.shape == x.shape, "Формы тензоров q, k и v должны совпадать с формой x"


        # Обратите внимание: это неэффективная реализация с циклом for.
        # Чтобы получить полный балл за это задание, перепишите её с учётом следующих требований:
        # 1) Не используйте for и другие циклы. Выполняйте вычисления параллельно с помощью векторизованных операций.
        # 2) Не используйте F.scaled_dot_product_attention. Реализуйте механизм внимания с помощью базовых операций PyTorch.
        head_outputs = []
        for head_index in range(self.num_heads):
            head_selector = range(self.head_size * head_index, self.head_size * (head_index + 1))

            head_queries = q[..., head_selector]
            head_keys = k[..., head_selector]
            head_values = v[..., head_selector]

            single_head_output = F.scaled_dot_product_attention(
                head_queries, head_keys, head_values,
                is_causal=True)
            # документация: https://pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html
            head_outputs.append(single_head_output)

        combined_head_outputs = torch.cat(head_outputs, dim=-1)
        return self.c_proj(combined_head_outputs)


## Реализуйте самовнимание с маской __без нормализации слоя и остаточных связей__ (6 баллов)

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MaskedSelfAttention(nn.Module):
    def __init__(self, dim: int, num_heads: int):
        super().__init__()
        self.c_attn = nn.Linear(dim, dim * 3)  # совместное вычисление запросов (query), ключей (key) и значений (value)
        self.c_proj = nn.Linear(dim, dim)  # выходная проекция
        self.dim = dim
        self.num_heads = num_heads
        self.head_size = dim // num_heads
        self.scale = self.head_size ** -0.5

    def forward(self, x):
        batch_size, seq_len, dim = x.shape

        # Вычислите матрицы запросов (Q), ключей (K) и значений (V).
        q, k, v = self.c_attn(x).split(self.dim, dim=-1)

        # Измените форму тензоров для многоголового внимания.
        q, k, v = (t.view(batch_size, seq_len, self.num_heads, self.head_size).transpose(1, 2) for t in (q, k, v))

        # Вычислите оценки внимания с помощью масштабированного скалярного произведения.
        scores = (q @ k.transpose(-2, -1)) * self.scale

        # Примените каузальную маску.
        future = torch.ones(seq_len, seq_len, dtype=torch.bool, device=x.device).triu(diagonal=1)
        scores = scores.masked_fill(future, float('-inf'))

        # Примените softmax, чтобы получить вероятности внимания.
        attn_probs = torch.softmax(scores, dim=-1)

        # Вычислите взвешенную сумму значений (V).
        head_outputs = attn_probs @ v

        # Верните контекстные эмбеддинги.
        combined = head_outputs.transpose(1, 2).reshape(batch_size, seq_len, dim)
        return self.c_proj(combined)

Проверьте, что ваша реализация работает корректно:

In [10]:
attn = MaskedSelfAttention(dim=768, num_heads=12)
attn.load_state_dict({'c_attn.weight': state_dict['h.0.attn.c_attn.weight'],
                      'c_attn.bias': state_dict['h.0.attn.c_attn.bias'],
                      'c_proj.weight': state_dict['h.0.attn.c_proj.weight'],
                      'c_proj.bias': state_dict['h.0.attn.c_proj.bias']})

torch.manual_seed(1337)
x = torch.randn(1, 10, 768)  # [batch_size, sequence_length, dim]
checksum = torch.sum(attn(x) * x)
assert abs(checksum.item() - 2703.6772) < 0.1, "Выход блока не совпадает с эталоном"
assert not torch.allclose(attn(x[:, (1, 0), :])[:, (1, 0), :], attn(x[:, (0, 1), :])), "При самовнимании с каузальной маской перестановка входных позиций не должна сводиться к такой же перестановке выходов"
print("Всё работает!")

Всё работает!


Теперь можно объединить блок внимания и полносвязный блок (MLP) в полноценный слой трансформера:

![Схема слоя трансформера](https://i.imgur.com/1sq2vHO.png)

## Соберите слой трансформера (2 балла)

In [11]:
class TransformerLayer(nn.Module):
    def __init__(self, dim: int, num_heads: int):
        super().__init__()
        self.ln_1 = nn.LayerNorm(dim)
        self.attn = MaskedSelfAttention(dim, num_heads)
        self.ln_2 = nn.LayerNorm(dim)
        self.mlp = FullyConnected(dim)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

In [12]:
layer = TransformerLayer(dim=768, num_heads=12)
layer.load_state_dict({k[5:]: v for k, v in state_dict.items() if k.startswith('h.10.')})
assert abs(torch.sum(layer(x) * x).item() - 9874.7383) < 0.1
print("Отличная работа!")

Отличная работа!


In [13]:
class GPT2(nn.Module):
    def __init__(self, vocab_size: int, dim: int, num_heads: int, num_layers: int, max_position_embeddings: int = 1024):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, dim)  # эмбеддинги токенов
        self.wpe = nn.Embedding(max_position_embeddings, dim)  # позиционные эмбеддинги
        self.ln_f = nn.LayerNorm(dim)   # итоговая нормализация слоя: после всех слоёв трансформера, до вычисления логитов

        self.h = nn.Sequential(*(TransformerLayer(dim, num_heads) for layer in range(num_layers)))

    def forward(self, input_ids):
        # input_ids.shape: [batch_size, sequence_length]; идентификаторы токенов типа int64
        position_ids = torch.arange(input_ids.shape[1], device=input_ids.device).unsqueeze(0)

        token_embeddings = self.wte(input_ids)
        position_embeddings = self.wpe(position_ids)
        full_embeddings = token_embeddings + position_embeddings

        transformer_output = self.h(full_embeddings)
        transformer_output_ln = self.ln_f(transformer_output)

        # Вычисляем логиты, используя матрицу эмбеддингов токенов как матрицу весов выходного линейного слоя.
        output_logits = transformer_output_ln @ self.wte.weight.T
        return output_logits


In [14]:
tokenizer = transformers.AutoTokenizer.from_pretrained('gpt2', add_prefix_space=True)
model = GPT2(vocab_size=50257, dim=768, num_heads=12, num_layers=12)
model.load_state_dict(state_dict)
model = model.to(device)

input_ids = tokenizer("A quick", return_tensors='pt')['input_ids'].to(device)

predicted_logits = model(input_ids)
most_likely_token_id = predicted_logits[:, -1].argmax().item()

print("Предсказание:", tokenizer.decode(most_likely_token_id))

Предсказание:  look


In [15]:
text = "The Fermi paradox "
tokens = tokenizer.encode(text)
print(end=tokenizer.decode(tokens))
line_length = len(tokenizer.decode(tokens))

temperature = 0.8
top_k = 50

for i in range(500):
    # Вычисляем логиты с помощью вашей модели.
    with torch.no_grad():
        logits = model(torch.as_tensor([tokens], device=device))

    # Выбираем следующий токен случайно, согласно предсказанным вероятностям.
    next_logits = logits[0, -1, :] / temperature
    top_values, top_indices = torch.topk(next_logits, top_k)
    probs = torch.softmax(top_values, dim=-1).cpu().numpy()
    next_token_index = top_indices[np.random.choice(top_k, p=probs)].item()
    #p_next = torch.softmax(logits[0, -1, :], dim=-1).data.cpu().numpy()
    #next_token_index = np.random.choice(len(p_next), p=p_next)

    tokens.append(int(next_token_index))
    print(end=tokenizer.decode(tokens[-1]))
    line_length += len(tokenizer.decode(tokens[-1]))
    if line_length > 120:
      line_length = 0
      print()



 The Fermi paradox  is a way of looking at a problem. The simple solution, as we've seen, is that it's time to start thinking
 about the problem. We all want to solve our problems. We want to avoid problems in our lives that might be hard to solve
 themselves by simply trying to solve them.
The problem is "why?" As my colleague, Christopher Schulz, explains in his article
 on the Fermi paradox, this is not the first time that Fermi has been caught off guard.  There have been many attempts to
 show that Fermi's solution is not right.  A few years ago, a group of scientists proposed that the Fermi paradox is not true
, and they then moved on to another problem, this time a more complicated one.
The simplest approach to the problem is to
 think in terms of a set of possible interactions, which may change during a given day.  To see these different groups of
 interactions, you must first think about every single interaction.  This process is called the "difficulty" problem.  The
 "difficul

### Как выполнить ту же задачу с помощью библиотеки transformers

In [16]:
tokenizer = transformers.AutoTokenizer.from_pretrained('gpt2', add_prefix_space=True)
model = transformers.AutoModelForCausalLM.from_pretrained('gpt2')
print('Сгенерированный текст:', tokenizer.decode(
    model.generate(
        **tokenizer("The Fermi paradox ", return_tensors='pt'),
        do_sample=True, max_new_tokens=50
    ).flatten().numpy()
))


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Сгенерированный текст:  The Fermi paradox  is perhaps the least interesting and least understood phenomena of the Universe .  Because it is a very small world, it cannot be considered a complete vacuum, and so it is not a vacuum we could find with some simple telescopes.  It
